# EDA 09: kiểm chứng ngưỡng stay point bằng nhãn phương tiện

**Giá trị khởi điểm:** D = 200 m, T = 30 phút (mặc định trong `src/stays/detect.py`, chốt 2026-11-07, A14), lấy từ bài gốc của thuật toán:
Li et al. (2008), *Mining user similarity based on location history*, ACM SIGSPATIAL GIS. Thuật toán ở Figure 2 ("Algorithm for stay point detection"); tham số ở mục 5.1.2 ("Parameter Selection"): *"we set timeThreh to 30 minutes and distThreh to 200 meters"*. Bài dùng dữ liệu đời đầu của chính dự án GeoLife (65 người, 6 tháng, ghi 2 giây/điểm). Bài nêu giá trị nhưng không giải thích vì sao chọn.

**Notebook này kiểm chứng giá trị khởi điểm đó trên dữ liệu của mình.** `labels.txt` (chỉ một phần user, xem mục 1) ghi các khoảng thời gian người dùng **đang di chuyển** (walk, bus, car...). Chạy stay point chỉ trên các điểm nằm trong những khoảng đó: mọi stay tìm được đều là **stay giả** (kẹt xe, chờ đèn, chờ tàu, đi bộ loanh quanh). Ngưỡng tốt phải cho rất ít stay giả.

**Giới hạn:**
- Chỉ đo được **nhận nhầm** (di chuyển bị coi là stay). Không đo được **bỏ sót** (lần ở lại thật bị mất), vì không có nhãn "đang ở yên".
- Chỉ một phần user có nhãn, và họ chỉ gắn nhãn những chuyến họ chọn.
- Một phần "stay giả" có thể là chờ thật nằm trong khoảng nhãn (người dùng bấm bắt đầu nhãn trước khi lên xe).

| Mục | Câu hỏi |
|---|---|
| 1 | Dữ liệu có nhãn gồm những gì, bao nhiêu giờ mỗi phương tiện? |
| 2 | Với từng (D, T), bao nhiêu thời gian di chuyển bị nhận nhầm là stay? |
| 3 | Ở giá trị khởi điểm, stay giả rơi vào phương tiện nào? |
| 4 | Stay giả là đứng yên, đi loanh quanh trong một khu, hay đang chạy xe? |

Các mục sau nằm ở `10_stay_anchor.ipynb` (mục 1–4, cách chọn điểm neo) và `11_gap_stay.ipynb` (mục 1–2, gap-stay).

**Cách chạy.** Cần `data/processed/clean` (chạy `clean_trajectory.py --save-clean data/processed/clean`) và `data/interim/labels`. Run All. Chỉ chạy trên các đoạn có nhãn nên nhanh (khoảng 2–5 phút). Kết quả từng user lưu ở `data/interim/eda_stays_labels/`; đổi code stay point thì xoá thư mục đó rồi chạy lại.

Code tính toán: `src/eda/stay_scan.py`.

> **Giai đoạn: TUNING (sau khi chia validation / test).** Chỉ đọc **nhóm validation** (`split.val_users()`). Không dùng nhóm test (D17, D19).

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from eda import stay_scan as ss
from eda import split

USERS_LIMIT = None     # ví dụ ["010", "085"] để chạy thử

pd.set_option("display.max_columns", 30)

users = ss.labeled_users()
if USERS_LIMIT:
    users = [u for u in users if u in set(USERS_LIMIT)]
print(f"{len(users)} user có nhãn và có dữ liệu đã lọc")

R = ss.run(users)
print(f"{len(R) // (len(ss.D_GRID) * len(ss.T_GRID_MIN)):,} đoạn có nhãn đủ 2 điểm")

42 user có nhãn và có dữ liệu đã lọc
[10/42] user 078 (17 giây)
[20/42] user 102 (17 giây)
[30/42] user 126 (18 giây)
[40/42] user 174 (51 giây)
[42/42] user 179 (51 giây)
5,845 đoạn có nhãn đủ 2 điểm


## 1. Dữ liệu có nhãn

Số đoạn và số giờ có điểm GPS của từng phương tiện (sau lọc, đã cắt ở khoảng trống > 30 phút). Phương tiện có ít giờ thì kết quả của nó ít tin cậy.

In [2]:
mot = R[(R["D"] == ss.CHOSEN[0]) & (R["T"] == ss.CHOSEN[1])]
bang14 = mot.groupby("mode").agg(số_đoạn=("hours", "size"), số_giờ=("hours", "sum"), số_user=("user", "nunique"))
bang14 = bang14.sort_values("số_giờ", ascending=False)
bang14.loc["tổng"] = [bang14["số_đoạn"].sum(), bang14["số_giờ"].sum(), mot["user"].nunique()]
display(bang14.round(1))

,số_đoạn,số_giờ,số_user
mode,,,
walk,2556.0,916.8,40.0
car,662.0,561.5,20.0
bus,983.0,487.7,27.0
bike,898.0,339.6,21.0
subway,341.0,164.3,19.0
train,67.0,116.8,13.0
taxi,320.0,106.5,16.0
airplane,8.0,17.3,2.0
boat,6.0,3.4,3.0


## 2. Tỷ lệ nhận nhầm theo (D, T)

- **% thời gian bị nhận nhầm**: tổng thời gian nằm trong stay giả / tổng thời gian di chuyển có nhãn. Chỉ số chính.
- **stay giả / 100 giờ di chuyển**: số lần nhận nhầm.
- **% user có stay giả**: tỷ lệ user có ít nhất một stay giả.

Hàng là D, cột là T. Ô D = 200, T = 30 là giá trị khởi điểm của bài gốc.

In [3]:
g = R.groupby(["D", "T"]).agg(hours=("hours", "sum"), false_hours=("false_hours", "sum"), n_false=("n_false", "sum"))
g["% thời gian bị nhận nhầm"] = 100 * g["false_hours"] / g["hours"]
g["stay giả / 100 giờ"] = 100 * g["n_false"] / g["hours"]
pu = R.groupby(["D", "T", "user"])["n_false"].sum().gt(0).groupby(["D", "T"]).mean() * 100
g["% user có stay giả"] = pu

for cot in ["% thời gian bị nhận nhầm", "stay giả / 100 giờ", "% user có stay giả"]:
    print(cot)
    display(g[cot].unstack("T").round(2).rename_axis(index="D (m)", columns="T (phút)"))

% thời gian bị nhận nhầm


T (phút),10,20,30
D (m),,,
100,16.25,9.54,6.29
200,23.48,14.97,10.39
300,29.12,19.26,13.88


stay giả / 100 giờ


T (phút),10,20,30
D (m),,,
100,51.11,17.64,8.58
200,69.00,26.95,13.55
300,82.36,32.99,17.45


% user có stay giả


T (phút),10,20,30
D (m),,,
100,78.05,68.29,56.10
200,90.24,75.61,63.41
300,92.68,78.05,65.85


## 3. Stay giả theo phương tiện (D = 200 m)

So T = 10, 20, 30 phút ở D = 200 m. Cột là % thời gian của phương tiện đó bị nhận nhầm là stay. Stay giả tập trung ở đâu cho biết kiểu dừng nào dễ bị nhầm: tàu, máy bay (chờ ở ga, sân bay), xe buýt (kẹt xe), đi bộ (loanh quanh trong khu nhỏ).

In [4]:
m = R[R["D"] == ss.CHOSEN[0]].groupby(["mode", "T"]).agg(hours=("hours", "sum"), fh=("false_hours", "sum"))
bang16 = (100 * m["fh"] / m["hours"]).unstack("T").round(2)
bang16.columns = [f"T = {c} phút (%)" for c in bang16.columns]
bang16["số giờ có nhãn"] = m["hours"].xs(ss.CHOSEN[1], level="T").round(0)
display(bang16.sort_values("số giờ có nhãn", ascending=False))

,T = 10 phút (%),T = 20 phút (%),T = 30 phút (%),số giờ có nhãn
mode,,,,
walk,48.12,33.60,23.72,917.0
car,9.51,4.18,2.52,561.0
bus,10.72,4.77,2.84,488.0
bike,16.03,9.20,6.43,340.0
subway,11.67,6.42,4.47,164.0
train,5.82,2.63,1.60,117.0
taxi,6.97,4.41,3.66,106.0
airplane,2.79,0.00,0.00,17.0
boat,17.74,12.46,0.00,3.0


## 4. Stay giả trông như thế nào (D = 200 m, T = 30 phút)

Nhãn "đang di chuyển" không có nghĩa là mọi lúc trong nhãn đều đang di chuyển: người dùng có thể gắn nhãn `walk` cho cả buổi dạo trong khuôn viên, công viên, trung tâm thương mại, hoặc quên kết thúc nhãn khi đã dừng. Mục này xem các stay giả để phân biệt:
- **đứng yên trong nhãn**: tốc độ trung vị rất thấp (< 1 km/h). Nhãn bao cả lúc dừng, thuật toán không sai.
- **đi loanh quanh trong một khu nhỏ**: tốc độ đi bộ nhưng `rad95_m` nhỏ (khoảng 100 m). Đây là "ở một nơi" theo nghĩa địa điểm (POI), dù nhãn ghi là đi bộ.
- **đang chạy xe**: tốc độ cao. Đây mới là nhận nhầm thật sự.

Lưu ý: nhiễu GPS khi đứng yên ở tần suất 1 điểm/giây có thể cho tốc độ biểu kiến 1–3 km/h, nên tốc độ thấp không phân biệt rõ đứng yên với đi rất chậm.

In [5]:
FS = ss.false_stay_details(users)
print(f"{len(FS)} stay giả, tổng {FS['stay_min'].sum() / 60:.0f} giờ")
bang17 = FS.groupby("mode").agg(số_stay=("stay_min", "size"), số_giờ=("stay_min", lambda x: x.sum() / 60),
                                tốc_độ_trung_vị_kmh=("med_kmh", "median"), quãng_đường_km=("path_km", "median"),
                                rad95_m=("rad95_m", "median"), độ_dài_nhãn_giờ=("lab_h", "median"),
                                chạm_đầu_cuối_nhãn=("at_label_edge", "mean"))
display(bang17.sort_values("số_giờ", ascending=False).round(2))

nhom = pd.cut(FS["med_kmh"], [-0.1, 1, 3, 6, np.inf], labels=["< 1 km/h", "1–3 km/h", "3–6 km/h", "> 6 km/h"])
display(FS.groupby(nhom, observed=False).agg(số_stay=("stay_min", "size"),
                                             số_giờ=("stay_min", lambda x: x.sum() / 60),
                                             rad95_m=("rad95_m", "median")).round(1))

368 stay giả, tổng 282 giờ


,số_stay,số_giờ,tốc_độ_trung_vị_kmh,quãng_đường_km,rad95_m,độ_dài_nhãn_giờ,chạm_đầu_cuối_nhãn
mode,,,,,,,
walk,279,217.44,2.09,0.91,117.02,2.48,0.21
bike,31,21.83,3.38,0.98,134.92,1.37,0.45
car,21,14.15,2.78,0.76,101.37,2.65,0.14
bus,18,13.86,3.55,1.11,104.14,1.99,0.33
subway,10,7.35,2.26,1.32,121.94,7.32,0.20
taxi,5,3.90,3.50,1.63,130.06,3.52,0.20
run,1,1.91,3.86,2.18,62.44,3.17,0.00
train,3,1.87,3.55,1.13,84.95,3.74,0.00


,số_stay,số_giờ,rad95_m
med_kmh,,,
< 1 km/h,74,55.4,104.7
1–3 km/h,143,119.9,119.4
3–6 km/h,140,100.4,117.3
> 6 km/h,11,6.7,188.6


## Tiêu chí chốt ngưỡng

1. **Giữ giá trị bài gốc** (D = 200 m, T = 30 phút) nếu tỷ lệ nhận nhầm thấp với các phương tiện xe (bus, car, taxi, train, subway; đề xuất dưới khoảng 5%), và phần lớn stay giả trong nhãn `walk` là đứng yên hoặc đi loanh quanh trong một khu nhỏ (mục 4), tức là không phải lỗi của ngưỡng.
2. **Giảm T** (20 hoặc 10 phút) chỉ khi tỷ lệ nhận nhầm vẫn thấp tương đương, vì T nhỏ giữ thêm các lần ghé ngắn (hữu ích cho POI).
3. **D:** không chọn D làm tỷ lệ nhận nhầm tăng rõ (D lớn gộp đi bộ chậm thành stay). D nhỏ hơn 200 m ít nhận nhầm hơn nhưng dễ làm vỡ stay thật; notebook này không đo được mặt đó, nên giữ 200 m nếu 200 m đã đủ thấp.
4. Ghi rõ giới hạn: chỉ đo nhận nhầm, không đo bỏ sót.

**Kết quả:** (42 user có nhãn, 5.845 đoạn, 2.716 giờ) ở D = 200 m, T = 30 phút, 10,39% thời gian di chuyển bị nhận là stay. Theo phương tiện xe: bus 2,84%, car 2,52%, taxi 3,66%, train 1,60%, subway 4,47%, đều dưới 5%. Đi bộ cao hơn hẳn (23,72%). Trong 368 stay giả (282 giờ), 279 stay thuộc nhãn `walk`; 74 stay có tốc độ trung vị dưới 1 km/h và 143 stay từ 1 đến 3 km/h (đứng yên hoặc đi rất chậm), chỉ 11 stay trên 6 km/h, và bán kính 95% khoảng 117 m (loanh quanh trong một khu nhỏ). Hạ T xuống 20 và 10 phút làm nhận nhầm tăng lên 14,97% và 23,48%. D = 100 m cho 6,29% và D = 300 m cho 13,88%.

**Quyết định:** D = 200 m, T = 30 phút (A14). Tiêu chí 1 đạt với các phương tiện xe; stay giả của `walk` chủ yếu là đứng yên hoặc đi loanh quanh trong một khu nhỏ, nên không coi là lỗi của ngưỡng. Không giảm T vì nhận nhầm tăng (tiêu chí 2). D = 100 m nhận nhầm ít hơn nhưng notebook không đo được mặt bỏ sót (tiêu chí 3), nên giữ 200 m, trùng bài gốc. Giới hạn: chỉ đo nhận nhầm, không đo bỏ sót; đây là quy ước, không phải tối ưu.